In [1]:
!pip install ripser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.5 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18104 sha256=d5e6daab3cf76b86e539835f320d8fa469e64735e680633fc347fdd89abda370
  Stored in directory: /root/.cache/pip/wheels/2a/fd/fe/f4b8fd82894e1d9e04040ef41dc5ae6eb7a8e9b0ef5a9402fe
Successfully built hopcroftkarp


# BVP-TPV

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd

from ripser import ripser
from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.spatial.distance import pdist
from scipy.interpolate import interp1d


# =============================================================================
# CONFIG
# =============================================================================
PKL_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/data/WESAD/pkl_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/TPV-csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS_LABEL = 700
FS_BVP = 64
FS_ACC = 32

WINDOW_SEC = 60
STEP_SEC = 60  # non-overlap

LABEL_BASELINE = 1
LABEL_STRESS = 2
LABEL_MEDITATION = 4

STATUS_MAP = {
    LABEL_BASELINE: 0,    # baseline
    LABEL_STRESS: 1,      # stress
    LABEL_MEDITATION: 2   # meditation
}
STATUS_NAME_MAP = {
    LABEL_BASELINE: "baseline",
    LABEL_STRESS: "stress",
    LABEL_MEDITATION: "meditation"
}

LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

# ---- Hard reject thresholds ----
FLATLINE_EPS = 1e-3
FLATLINE_MIN_SEC = 1.0
CLIPPING_RATIO_MAX = 0.10
MIN_PEAKS_PER_MIN = 30

# ---- Relaxed QC thresholds ----
VBR_MIN = 0.70
TC_MIN = 0.75
IBI_PASS_MIN = 0.80
MOTION_PERCENTILE = 80  # subject-personalized threshold

# ---- Peak / beat extraction ----
MIN_HR_BPM = 40
MAX_HR_BPM = 180
MIN_PEAK_DISTANCE_SEC = 60.0 / MAX_HR_BPM   # ~0.333 s
MAX_PEAK_DISTANCE_SEC = 60.0 / MIN_HR_BPM   # 1.5 s
IBI_MIN_SEC = 0.30
IBI_MAX_SEC = 1.50

BEAT_BEFORE_SEC = 0.25
BEAT_AFTER_SEC = 0.45
BEAT_RESAMPLED_LEN = 100

# ---- Motion artifact thresholds (new) ----
ALLOW_MISSING = False
ACC_DIFF_THRESH = 0.05
PSD_CORR_THRESH = 0.90
PSD_FMIN = 0.5
PSD_FMAX = 8.0

# selected TPV features only
SELECTED_TPV_INDICES = [22, 9, 4, 2, 0, 28, 8, 19, 14, 26, 11]

TPV_FEATURE_NAMES = [
    "birth_mean",                  # 0
    "birth_std",                   # 1
    "death_mean",                  # 2
    "death_std",                   # 3
    "lifetime_mean",               # 4
    "lifetime_std",                # 5
    "lifetime_max",                # 6
    "lifetime_min",                # 7
    "lifetime_median",             # 8
    "lifetime_iqr",                # 9
    "lifetime_skew",               # 10
    "lifetime_kurtosis",           # 11
    "birth_max",                   # 12
    "birth_min",                   # 13
    "birth_median",                # 14
    "birth_skew",                  # 15
    "birth_kurtosis",              # 16
    "death_max",                   # 17
    "death_min",                   # 18
    "death_median",                # 19
    "death_skew",                  # 20
    "death_kurtosis",              # 21
    "num_lifetimes",               # 22
    "top1_lifetime",               # 23
    "top2_lifetime",               # 24
    "lifetime_max_div_min",        # 25
    "ph_entropy",                  # 26
    "betti_entropy",               # 27
    "avg_persistence_distance",    # 28
    "gini_index",                  # 29
    "lifetime_variance",           # 30
    "persistent_image_energy",     # 31
]
N_FEATURES = len(TPV_FEATURE_NAMES)


# =============================================================================
# Basic utils
# =============================================================================
def fill_nan_with_median(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def fill_nan_acc_with_median(acc_xyz: np.ndarray) -> np.ndarray:
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32).copy()
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3:
        return acc_xyz
    for c in range(3):
        ch = acc_xyz[:, c]
        med = np.nanmedian(ch)
        if np.isnan(med):
            med = 0.0
        ch = np.nan_to_num(ch, nan=med, posinf=med, neginf=med)
        acc_xyz[:, c] = ch
    return acc_xyz


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_skew(x: np.ndarray) -> float:
    if len(x) > 2 and np.std(x) > 1e-12:
        v = skew(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_kurtosis(x: np.ndarray) -> float:
    if len(x) > 3 and np.std(x) > 1e-12:
        v = kurtosis(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def clamp01(x):
    return float(np.clip(x, 0.0, 1.0))


def compute_missing_ratio(x: np.ndarray) -> float:
    x = np.asarray(x)
    return float(np.mean(~np.isfinite(x)))


# =============================================================================
# TPV extraction
# =============================================================================
def extract_tpv_features(sig_1d: np.ndarray) -> np.ndarray:
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)

    if len(sig) < 3:
        return np.zeros(N_FEATURES, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(N_FEATURES, dtype=np.float32)

    sig = (sig - float(np.mean(sig))) / (s + 1e-8)

    X = np.stack([sig[:-1], sig[1:]], axis=1)
    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]

    if H1.size == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births = births[mask]
    deaths = deaths[mask]
    lifetimes = lifetimes[mask]

    if len(lifetimes) == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)
    lifetime_sum = float(np.sum(lifetimes))
    lifetime_ratio = lifetimes / (lifetime_sum + 1e-8)
    ph_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin = float(np.min(births))
    bmax = float(np.max(births))
    if bmax - bmin < 1e-8:
        betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        betti_entropy = float(entropy(hist + 1e-10))

    avg_persistence_distance = (
        float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0
    )

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (
            2 * np.sum(np.arange(1, n + 1) * lifetimes_sorted)
        ) / (n * np.sum(lifetimes_sorted)) - (n + 1) / n
        gini_index = float(gini)
    else:
        gini_index = 0.0

    lifetime_variance = float(np.var(lifetimes))
    persistent_image_energy = float(np.sum(lifetimes ** 2))

    feats = [
        float(np.mean(births)),                                  # 0
        float(np.std(births)),                                   # 1
        float(np.mean(deaths)),                                  # 2
        float(np.std(deaths)),                                   # 3
        float(np.mean(lifetimes)),                               # 4
        float(np.std(lifetimes)),                                # 5
        float(np.max(lifetimes)),                                # 6
        float(np.min(lifetimes)),                                # 7
        float(np.median(lifetimes)),                             # 8
        float(iqr(lifetimes)),                                   # 9
        safe_skew(lifetimes),                                    # 10
        safe_kurtosis(lifetimes),                                # 11
        float(np.max(births)),                                   # 12
        float(np.min(births)),                                   # 13
        float(np.median(births)),                                # 14
        safe_skew(births),                                       # 15
        safe_kurtosis(births),                                   # 16
        float(np.max(deaths)),                                   # 17
        float(np.min(deaths)),                                   # 18
        float(np.median(deaths)),                                # 19
        safe_skew(deaths),                                       # 20
        safe_kurtosis(deaths),                                   # 21
        float(len(lifetimes)),                                   # 22
        float(lifetimes_sorted[-1]),                             # 23
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,  # 24
        float(np.max(lifetimes) / (np.min(lifetimes) + 1e-8)),   # 25
        ph_entropy,                                              # 26
        betti_entropy,                                           # 27
        avg_persistence_distance,                                # 28
        gini_index,                                              # 29
        lifetime_variance,                                       # 30
        persistent_image_energy,                                 # 31
    ]
    return np.asarray(feats, dtype=np.float32)


# =============================================================================
# Find target blocks: first baseline, first stress, last meditation
# =============================================================================
def find_target_blocks(labels: np.ndarray):
    labels = np.asarray(labels).reshape(-1)
    if len(labels) == 0:
        return {}

    segments = []
    start = 0
    cur = labels[0]
    for i in range(1, len(labels)):
        if labels[i] != cur:
            segments.append((int(cur), start, i))
            start = i
            cur = labels[i]
    segments.append((int(cur), start, len(labels)))

    baseline_seg = None
    stress_seg = None
    meditation_seg = None

    for lab, s, e in segments:
        if lab == LABEL_BASELINE:
            baseline_seg = (s, e)
            break

    for lab, s, e in segments:
        if lab == LABEL_STRESS:
            stress_seg = (s, e)
            break

    for lab, s, e in reversed(segments):
        if lab == LABEL_MEDITATION:
            meditation_seg = (s, e)
            break

    out = {}
    if baseline_seg is not None:
        out[LABEL_BASELINE] = baseline_seg
    if stress_seg is not None:
        out[LABEL_STRESS] = stress_seg
    if meditation_seg is not None:
        out[LABEL_MEDITATION] = meditation_seg
    return out


def window_fully_in_block(start_l: int, end_l: int, label_major: int, target_blocks: dict) -> bool:
    if label_major not in target_blocks:
        return False
    blk_s, blk_e = target_blocks[label_major]
    return (start_l >= blk_s) and (end_l <= blk_e)


# =============================================================================
# QC metric helpers
# =============================================================================
def is_flatline(sig: np.ndarray, fs: int, eps: float = FLATLINE_EPS, min_sec: float = FLATLINE_MIN_SEC) -> bool:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    run_len = int(round(fs * min_sec))
    if len(sig) < run_len:
        return False
    ds = np.abs(np.diff(sig))
    low_change = ds < eps
    cnt = 0
    for v in low_change:
        if v:
            cnt += 1
            if cnt >= run_len - 1:
                return True
        else:
            cnt = 0
    return False


def clipping_ratio(sig: np.ndarray) -> float:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    if len(sig) == 0:
        return 1.0
    lo = np.percentile(sig, 1)
    hi = np.percentile(sig, 99)
    if hi - lo < 1e-8:
        return 1.0
    near_lo = sig <= (lo + 0.01 * (hi - lo))
    near_hi = sig >= (hi - 0.01 * (hi - lo))
    return float(np.mean(near_lo | near_hi))


def detect_ppg_peaks(sig: np.ndarray, fs: int):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    distance = max(1, int(round(MIN_PEAK_DISTANCE_SEC * fs)))
    prom = max(0.10, 0.20 * float(np.std(sig)))
    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def compute_motion_score(acc_xyz: np.ndarray) -> float:
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32)
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3 or len(acc_xyz) < 2:
        return 1.0
    mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    dmag = np.abs(np.diff(mag))
    score = np.median(dmag) + 0.5 * np.std(dmag)
    return float(score)


def compute_acc_diff_metrics(acc_xyz: np.ndarray):
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32)
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3 or len(acc_xyz) < 2:
        return {
            "acc_diff_mean": np.nan,
            "acc_diff_exceed_ratio": np.nan,
            "acc_diff_flag": 1
        }

    mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    dmag = np.abs(np.diff(mag))

    acc_diff_mean = float(np.mean(dmag)) if len(dmag) > 0 else 0.0
    acc_diff_exceed_ratio = float(np.mean(dmag > ACC_DIFF_THRESH)) if len(dmag) > 0 else 0.0
    acc_diff_flag = int(acc_diff_mean > ACC_DIFF_THRESH)

    return {
        "acc_diff_mean": acc_diff_mean,
        "acc_diff_exceed_ratio": acc_diff_exceed_ratio,
        "acc_diff_flag": acc_diff_flag
    }


def _safe_psd_1d(x: np.ndarray, fs: int, fmin: float, fmax: float):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) < 8:
        return None, None

    x = x[np.isfinite(x)]
    if len(x) < 8:
        return None, None

    if np.std(x) < 1e-8:
        return None, None

    nperseg = min(256, len(x))
    f, pxx = welch(x, fs=fs, nperseg=nperseg)

    mask = (f >= fmin) & (f <= min(fmax, fs / 2 - 1e-6))
    if np.sum(mask) < 4:
        return None, None

    return f[mask], pxx[mask]


def _corr_on_common_freq(f_ref, p_ref, f_other, p_other):
    if f_ref is None or f_other is None:
        return np.nan

    p_other_interp = np.interp(f_ref, f_other, p_other)

    a = np.log1p(p_ref)
    b = np.log1p(p_other_interp)

    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan

    r = np.corrcoef(a, b)[0, 1]
    return float(r) if np.isfinite(r) else np.nan


def compute_bvp_acc_psd_corr(seg_bvp_bp: np.ndarray, seg_acc_xyz: np.ndarray,
                             fs_bvp: int, fs_acc: int,
                             fmin: float = PSD_FMIN, fmax: float = PSD_FMAX):
    out = {
        "psd_corr_x": np.nan,
        "psd_corr_y": np.nan,
        "psd_corr_z": np.nan,
        "psd_corr_mag": np.nan,
        "psd_corr_max": np.nan,
        "psd_corr_flag": 1
    }

    seg_bvp_bp = np.asarray(seg_bvp_bp, dtype=np.float32).reshape(-1)
    acc_xyz = np.asarray(seg_acc_xyz, dtype=np.float32)

    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3:
        return out

    f_bvp, p_bvp = _safe_psd_1d(seg_bvp_bp, fs_bvp, fmin, fmax)
    if f_bvp is None:
        return out

    acc_mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))

    corr_list = []
    names_and_signals = [
        ("x", acc_xyz[:, 0]),
        ("y", acc_xyz[:, 1]),
        ("z", acc_xyz[:, 2]),
        ("mag", acc_mag),
    ]

    for name, sig in names_and_signals:
        f_acc, p_acc = _safe_psd_1d(sig, fs_acc, fmin, fmax)
        r = _corr_on_common_freq(f_bvp, p_bvp, f_acc, p_acc)
        out[f"psd_corr_{name}"] = r
        if np.isfinite(r):
            corr_list.append(r)

    if len(corr_list) == 0:
        return out

    psd_corr_max = float(np.max(corr_list))
    out["psd_corr_max"] = psd_corr_max
    out["psd_corr_flag"] = int(psd_corr_max >= PSD_CORR_THRESH)

    return out


def extract_beats(sig: np.ndarray, peaks: np.ndarray, fs: int):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    beats = []
    before = int(round(BEAT_BEFORE_SEC * fs))
    after = int(round(BEAT_AFTER_SEC * fs))
    target_x = np.linspace(0, 1, BEAT_RESAMPLED_LEN)

    for p in peaks:
        s = p - before
        e = p + after
        if s < 0 or e > len(sig):
            continue
        beat = sig[s:e].copy()
        if len(beat) < 5:
            continue
        src_x = np.linspace(0, 1, len(beat))
        f = interp1d(src_x, beat, kind="linear")
        beat_rs = f(target_x).astype(np.float32)
        beats.append(beat_rs)

    if len(beats) == 0:
        return np.empty((0, BEAT_RESAMPLED_LEN), dtype=np.float32)
    return np.stack(beats, axis=0)


def median_template_corr(beats: np.ndarray) -> float:
    if beats.ndim != 2 or len(beats) < 2:
        return 0.0
    template = np.median(beats, axis=0)
    tc_list = []
    for b in beats:
        sb = np.std(b)
        st = np.std(template)
        if sb < 1e-8 or st < 1e-8:
            tc_list.append(0.0)
        else:
            r = np.corrcoef(b, template)[0, 1]
            if np.isfinite(r):
                tc_list.append(float(r))
            else:
                tc_list.append(0.0)
    return float(np.median(tc_list)) if len(tc_list) > 0 else 0.0


def compute_ibi_metrics(peaks: np.ndarray, fs: int):
    peaks = np.asarray(peaks, dtype=np.int64)
    if len(peaks) < 3:
        return {
            "n_peaks": int(len(peaks)),
            "ibi_pass_ratio": 0.0,
            "valid_beat_ratio": 0.0,
            "ibi_plausibility": 0.0,
            "ibi_mean_sec": np.nan,
        }

    ibi = np.diff(peaks) / float(fs)
    plausible = (ibi >= IBI_MIN_SEC) & (ibi <= IBI_MAX_SEC)
    ibi_pass_ratio = float(np.mean(plausible)) if len(ibi) > 0 else 0.0

    if len(ibi) >= 3:
        med_ibi = np.median(ibi)
        valid = plausible & (np.abs(ibi - med_ibi) <= 0.30 * (med_ibi + 1e-8))
        vbr = float(np.mean(valid))
    else:
        vbr = ibi_pass_ratio

    return {
        "n_peaks": int(len(peaks)),
        "ibi_pass_ratio": ibi_pass_ratio,
        "valid_beat_ratio": vbr,
        "ibi_plausibility": ibi_pass_ratio,
        "ibi_mean_sec": float(np.mean(ibi)) if len(ibi) > 0 else np.nan,
    }


def notch_foot_peak_detectability_score(beats: np.ndarray) -> float:
    if beats.ndim != 2 or len(beats) == 0:
        return 0.0

    scores = []
    for b in beats:
        foot = np.min(b[:max(5, len(b)//4)])
        peak = np.max(b)
        amp = peak - foot
        dyn = np.max(b) - np.min(b) + 1e-8
        s = amp / dyn
        scores.append(clamp01(s))

    return float(np.median(scores)) if len(scores) > 0 else 0.0


def compute_sqi_scores(vbr, tc, ibi_plausibility, motion_score, morph_score):
    motion_penalty = clamp01(motion_score)

    sqi_stress_basic = (
        0.40 * clamp01(vbr)
        + 0.30 * clamp01(tc)
        + 0.20 * clamp01(ibi_plausibility)
        - 0.10 * motion_penalty
    )

    sqi_stress_morph = (
        0.30 * clamp01(vbr)
        + 0.25 * clamp01(tc)
        + 0.20 * clamp01(ibi_plausibility)
        + 0.25 * clamp01(morph_score)
        - motion_penalty
    )

    return float(sqi_stress_basic), float(sqi_stress_morph), motion_penalty


def evaluate_window_quality(seg_bvp_raw: np.ndarray, seg_acc_xyz: np.ndarray, fs_bvp: int, fs_acc: int):
    """
    Returns preprocessing output + QC metrics.
    기존 QC는 유지하고, motion artifact 제거 조건만 추가.
    """
    out = {}

    # -------------------------------------------------------------------------
    # Missing check (new)
    # -------------------------------------------------------------------------
    out["bvp_missing_ratio"] = compute_missing_ratio(seg_bvp_raw)
    out["acc_missing_ratio"] = compute_missing_ratio(seg_acc_xyz)
    out["missing_flag"] = int(
        (out["bvp_missing_ratio"] > 0.0) or
        (out["acc_missing_ratio"] > 0.0)
    ) if not ALLOW_MISSING else 0

    # BVP/ACC 보정본 생성
    seg_bvp = fill_nan_with_median(seg_bvp_raw)
    seg_acc_xyz = fill_nan_acc_with_median(seg_acc_xyz)

    # -------------------------------------------------------------------------
    # BVP preprocess
    # -------------------------------------------------------------------------
    seg_bvp_bp = bandpass_filter(seg_bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_bvp_z = zscore_1d(seg_bvp_bp)

    # -------------------------------------------------------------------------
    # Existing QC
    # -------------------------------------------------------------------------
    out["flatline_flag"] = int(is_flatline(seg_bvp_bp, fs=fs_bvp))
    out["clipping_ratio"] = clipping_ratio(seg_bvp_bp)
    out["clipping_flag"] = int(out["clipping_ratio"] > CLIPPING_RATIO_MAX)

    peaks, props = detect_ppg_peaks(seg_bvp_z, fs=fs_bvp)
    out["n_peaks"] = int(len(peaks))
    out["peak_sparse_flag"] = int(len(peaks) < MIN_PEAKS_PER_MIN)

    ibi_info = compute_ibi_metrics(peaks, fs_bvp)
    out.update(ibi_info)

    beats = extract_beats(seg_bvp_z, peaks, fs_bvp)
    tc = median_template_corr(beats)
    morph_score = notch_foot_peak_detectability_score(beats)
    motion_score_raw = compute_motion_score(seg_acc_xyz)

    out["median_template_corr"] = float(tc)
    out["notch_foot_peak_detectability"] = float(morph_score)
    out["motion_score_raw"] = float(motion_score_raw)

    sqi_basic, sqi_morph, motion_penalty = compute_sqi_scores(
        vbr=out["valid_beat_ratio"],
        tc=tc,
        ibi_plausibility=out["ibi_plausibility"],
        motion_score=motion_score_raw,
        morph_score=morph_score
    )

    out["motion_score"] = float(motion_penalty)
    out["SQI_stress_basic"] = float(sqi_basic)
    out["SQI_stress_morph"] = float(sqi_morph)

    # -------------------------------------------------------------------------
    # Motion artifact QC (new)
    # -------------------------------------------------------------------------
    acc_diff_info = compute_acc_diff_metrics(seg_acc_xyz)
    out.update(acc_diff_info)

    psd_corr_info = compute_bvp_acc_psd_corr(
        seg_bvp_bp=seg_bvp_bp,
        seg_acc_xyz=seg_acc_xyz,
        fs_bvp=fs_bvp,
        fs_acc=fs_acc,
        fmin=PSD_FMIN,
        fmax=PSD_FMAX
    )
    out.update(psd_corr_info)

    # -------------------------------------------------------------------------
    # Hard reject
    # -------------------------------------------------------------------------
    hard_reject = (
        (out["missing_flag"] == 1) or
        (out["flatline_flag"] == 1) or
        (out["clipping_flag"] == 1) or
        (out["peak_sparse_flag"] == 1) or
        (out["acc_diff_flag"] == 1) or
        (out["psd_corr_flag"] == 1)
    )
    out["hard_reject"] = int(hard_reject)

    return seg_bvp_z, out


# =============================================================================
# Build table for one subject
# =============================================================================
def build_subject_windows_table(pkl_path: str, subject_name: str) -> pd.DataFrame:
    with open(pkl_path, "rb") as f:
        s = pickle.load(f, encoding="latin1")

    bvp = np.asarray(s["signal"]["wrist"]["BVP"]).reshape(-1).astype(np.float32)
    acc = np.asarray(s["signal"]["wrist"]["ACC"]).astype(np.float32)
    labels = np.asarray(s["label"]).reshape(-1).astype(np.int64)

    dur_wrist = len(bvp) / FS_BVP
    labels = labels[: int(dur_wrist * FS_LABEL)]

    bvp = fill_nan_with_median(bvp)

    target_blocks = find_target_blocks(labels)
    if len(target_blocks) == 0:
        return pd.DataFrame()

    Wl = int(WINDOW_SEC * FS_LABEL)
    Sl = int(STEP_SEC * FS_LABEL)
    Wb = int(WINDOW_SEC * FS_BVP)
    Wa = int(WINDOW_SEC * FS_ACC)

    rows = []
    window_counter = 0

    for start_l in range(0, len(labels) - Wl + 1, Sl):
        end_l = start_l + Wl
        win_lab = labels[start_l:end_l]

        binc = np.bincount(win_lab)
        maj = int(np.argmax(binc))
        maj_ratio = float((win_lab == maj).mean())

        if maj not in [LABEL_BASELINE, LABEL_STRESS, LABEL_MEDITATION]:
            continue
        if maj_ratio < 0.95:
            continue
        if not window_fully_in_block(start_l, end_l, maj, target_blocks):
            continue

        t0 = start_l / FS_LABEL
        t1 = t0 + WINDOW_SEC

        start_b = int(round(t0 * FS_BVP))
        end_b = start_b + Wb
        start_a = int(round(t0 * FS_ACC))
        end_a = start_a + Wa

        if end_b > len(bvp) or end_a > len(acc):
            continue

        seg_bvp_raw = bvp[start_b:end_b]
        seg_acc_xyz = acc[start_a:end_a]

        if len(seg_bvp_raw) != Wb or len(seg_acc_xyz) != Wa:
            continue

        seg_bvp_for_tpv, qc_info = evaluate_window_quality(seg_bvp_raw, seg_acc_xyz, FS_BVP, FS_ACC)
        tpv_full = extract_tpv_features(seg_bvp_for_tpv)

        window_counter += 1

        row = {
            "subject": subject_name,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "status": STATUS_MAP[maj],
            "status_name": STATUS_NAME_MAP[maj],
            "label_major": maj,
            "t_start_sec": float(t0),
            "t_end_sec": float(t1),
            "major_ratio": maj_ratio,
        }

        for k, v in qc_info.items():
            row[k] = v

        for idx in SELECTED_TPV_INDICES:
            row[f"TPV{idx}"] = float(tpv_full[idx])

        rows.append(row)

    return pd.DataFrame(rows)


# =============================================================================
# Build all subjects
# =============================================================================
def build_all_subjects_windows_table(pkl_dir: str) -> pd.DataFrame:
    pkl_list = sorted(glob.glob(os.path.join(pkl_dir, "S*.pkl")))
    all_dfs = []

    print(f"[INFO] Found {len(pkl_list)} pkl files")

    for pkl_path in pkl_list:
        subject_name = os.path.splitext(os.path.basename(pkl_path))[0]
        print(f"\n[INFO] Processing {subject_name} ...")
        try:
            df_sub = build_subject_windows_table(pkl_path, subject_name)
            print(f"[INFO] {subject_name}: extracted {len(df_sub)} target windows")
            if len(df_sub) > 0:
                all_dfs.append(df_sub)
        except Exception as e:
            print(f"[WARN] Failed on {subject_name}: {e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    return pd.concat(all_dfs, axis=0, ignore_index=True)


# =============================================================================
# Personalized QC pass/fail using MR <= subject 80 percentile
# =============================================================================
def apply_relaxed_personalized_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    motion_thr_map = (
        df.groupby("subject")["motion_score"]
        .quantile(MOTION_PERCENTILE / 100.0)
        .to_dict()
    )

    df["motion_threshold_subject_p80"] = df["subject"].map(motion_thr_map)

    df["qc_relaxed_pass"] = (
        (df["hard_reject"] == 0) &
        (df["valid_beat_ratio"] >= VBR_MIN) &
        (df["median_template_corr"] >= TC_MIN) &
        (df["ibi_pass_ratio"] >= IBI_PASS_MIN) &
        (df["motion_score"] <= df["motion_threshold_subject_p80"])
    ).astype(int)

    return df


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    df_all = build_all_subjects_windows_table(PKL_DIR)

    print("\n[INFO] final raw shape:", df_all.shape)
    if len(df_all) == 0:
        print("[WARN] No target windows extracted.")
        raise SystemExit

    df_all = apply_relaxed_personalized_qc(df_all)

    # column order
    base_cols = [
        "subject", "window", "window_index",
        "status", "status_name", "label_major",
        "t_start_sec", "t_end_sec", "major_ratio"
    ]

    qc_cols = [
        "bvp_missing_ratio", "acc_missing_ratio", "missing_flag",
        "flatline_flag", "clipping_ratio", "clipping_flag",
        "n_peaks", "peak_sparse_flag",
        "ibi_pass_ratio", "valid_beat_ratio", "ibi_plausibility", "ibi_mean_sec",
        "median_template_corr", "notch_foot_peak_detectability",

        "acc_diff_mean", "acc_diff_exceed_ratio", "acc_diff_flag",

        "psd_corr_x", "psd_corr_y", "psd_corr_z", "psd_corr_mag",
        "psd_corr_max", "psd_corr_flag",

        "motion_score_raw", "motion_score",
        "SQI_stress_basic", "SQI_stress_morph",
        "hard_reject", "motion_threshold_subject_p80", "qc_relaxed_pass"
    ]

    feat_cols = [f"TPV{idx}" for idx in SELECTED_TPV_INDICES]

    final_cols = base_cols + qc_cols + feat_cols
    df_all = df_all[final_cols].copy()

    # 1) save all target windows (without applying QC filter)
    csv_all = os.path.join(OUTPUT_DIR, "noQC.csv")
    df_all.to_csv(csv_all, index=False)
    print(f"[INFO] Saved no-QC CSV: {csv_all}")

    # 2) save only QC-passed windows
    df_qc = df_all[df_all["qc_relaxed_pass"] == 1].copy()
    csv_qc = os.path.join(OUTPUT_DIR, "withQC.csv")
    df_qc.to_csv(csv_qc, index=False)
    print(f"[INFO] Saved QC CSV: {csv_qc}")

    # 3) subject/status summary
    summary = (
        df_all.groupby(["subject", "status_name"])
        .agg(
            n_total=("window", "count"),
            n_qc_pass=("qc_relaxed_pass", "sum")
        )
        .reset_index()
    )
    summary["retention_ratio"] = summary["n_qc_pass"] / summary["n_total"]

    summary_csv = os.path.join(OUTPUT_DIR, "summary.csv")
    summary.to_csv(summary_csv, index=False)
    print(f"[INFO] Saved QC retention summary: {summary_csv}")

    print("\n[INFO] Status counts (all target windows)")
    print(df_all["status_name"].value_counts())

    print("\n[INFO] Status counts (QC passed)")
    print(df_qc["status_name"].value_counts())

    print("\n[INFO] Subject x status retention summary")
    print(summary.to_string(index=False))

[INFO] Found 15 pkl files

[INFO] Processing S10 ...
[INFO] S10: extracted 36 target windows

[INFO] Processing S11 ...
[INFO] S11: extracted 35 target windows

[INFO] Processing S13 ...
[INFO] S13: extracted 35 target windows

[INFO] Processing S14 ...
[INFO] S14: extracted 35 target windows

[INFO] Processing S15 ...
[INFO] S15: extracted 35 target windows

[INFO] Processing S16 ...
[INFO] S16: extracted 35 target windows

[INFO] Processing S17 ...
[INFO] S17: extracted 36 target windows

[INFO] Processing S2 ...
[INFO] S2: extracted 33 target windows

[INFO] Processing S3 ...
[INFO] S3: extracted 34 target windows

[INFO] Processing S4 ...
[INFO] S4: extracted 34 target windows

[INFO] Processing S5 ...
[INFO] S5: extracted 33 target windows

[INFO] Processing S6 ...
[INFO] S6: extracted 34 target windows

[INFO] Processing S7 ...
[INFO] S7: extracted 35 target windows

[INFO] Processing S8 ...
[INFO] S8: extracted 36 target windows

[INFO] Processing S9 ...
[INFO] S9: extracted 32 

# EDA-TPV

In [2]:
import os
import glob
import pickle
import numpy as np
import pandas as pd

from ripser import ripser
from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.spatial.distance import pdist
from scipy.interpolate import interp1d


# =============================================================================
# CONFIG
# =============================================================================
PKL_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/data/WESAD/pkl_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/TPV-csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS_LABEL = 700
FS_EDA = 4
FS_ACC = 32

WINDOW_SEC = 60
STEP_SEC = 60  # non-overlap

LABEL_BASELINE = 1
LABEL_STRESS = 2
LABEL_MEDITATION = 4

STATUS_MAP = {
    LABEL_BASELINE: 0,    # baseline
    LABEL_STRESS: 1,      # stress
    LABEL_MEDITATION: 2   # meditation
}
STATUS_NAME_MAP = {
    LABEL_BASELINE: "baseline",
    LABEL_STRESS: "stress",
    LABEL_MEDITATION: "meditation"
}

# ---- EDA filter ----
LOWCUT = 0.05
HIGHCUT = 1.0
FILTER_ORDER = 4

# ---- Hard reject thresholds ----
FLATLINE_EPS = 1e-4
FLATLINE_MIN_SEC = 3.0
CLIPPING_RATIO_MAX = 0.15
MIN_PEAKS_PER_MIN = 2   # EDA라 BVP보다 훨씬 낮게 둠

# ---- Relaxed QC thresholds ----
VBR_MIN = 0.30
TC_MIN = 0.30
IBI_PASS_MIN = 0.30
MOTION_PERCENTILE = 80  # subject-personalized threshold

# ---- Peak / pseudo-event extraction ----
MIN_EVENT_DISTANCE_SEC = 1.0
MAX_EVENT_DISTANCE_SEC = 10.0
IBI_MIN_SEC = 0.5
IBI_MAX_SEC = 10.0

BEAT_BEFORE_SEC = 1.0
BEAT_AFTER_SEC = 3.0
BEAT_RESAMPLED_LEN = 100

# ---- Motion artifact thresholds ----
ALLOW_MISSING = False
ACC_DIFF_THRESH = 0.05
PSD_CORR_THRESH = 0.90
PSD_FMIN = 0.05
PSD_FMAX = 1.0

# selected TPV features only
SELECTED_TPV_INDICES = [22, 9, 4, 2, 0, 28, 8, 19, 14, 26, 11]

TPV_FEATURE_NAMES = [
    "birth_mean",                  # 0
    "birth_std",                   # 1
    "death_mean",                  # 2
    "death_std",                   # 3
    "lifetime_mean",               # 4
    "lifetime_std",                # 5
    "lifetime_max",                # 6
    "lifetime_min",                # 7
    "lifetime_median",             # 8
    "lifetime_iqr",                # 9
    "lifetime_skew",               # 10
    "lifetime_kurtosis",           # 11
    "birth_max",                   # 12
    "birth_min",                   # 13
    "birth_median",                # 14
    "birth_skew",                  # 15
    "birth_kurtosis",              # 16
    "death_max",                   # 17
    "death_min",                   # 18
    "death_median",                # 19
    "death_skew",                  # 20
    "death_kurtosis",              # 21
    "num_lifetimes",               # 22
    "top1_lifetime",               # 23
    "top2_lifetime",               # 24
    "lifetime_max_div_min",        # 25
    "ph_entropy",                  # 26
    "betti_entropy",               # 27
    "avg_persistence_distance",    # 28
    "gini_index",                  # 29
    "lifetime_variance",           # 30
    "persistent_image_energy",     # 31
]
N_FEATURES = len(TPV_FEATURE_NAMES)


# =============================================================================
# Basic utils
# =============================================================================
def fill_nan_with_median(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def fill_nan_acc_with_median(acc_xyz: np.ndarray) -> np.ndarray:
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32).copy()
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3:
        return acc_xyz
    for c in range(3):
        ch = acc_xyz[:, c]
        med = np.nanmedian(ch)
        if np.isnan(med):
            med = 0.0
        ch = np.nan_to_num(ch, nan=med, posinf=med, neginf=med)
        acc_xyz[:, c] = ch
    return acc_xyz


def bandpass_filter(sig, fs=4, low=0.05, high=1.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_skew(x: np.ndarray) -> float:
    if len(x) > 2 and np.std(x) > 1e-12:
        v = skew(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_kurtosis(x: np.ndarray) -> float:
    if len(x) > 3 and np.std(x) > 1e-12:
        v = kurtosis(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def clamp01(x):
    return float(np.clip(x, 0.0, 1.0))


def compute_missing_ratio(x: np.ndarray) -> float:
    x = np.asarray(x)
    return float(np.mean(~np.isfinite(x)))


# =============================================================================
# TPV extraction
# =============================================================================
def extract_tpv_features(sig_1d: np.ndarray) -> np.ndarray:
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)

    if len(sig) < 3:
        return np.zeros(N_FEATURES, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(N_FEATURES, dtype=np.float32)

    sig = (sig - float(np.mean(sig))) / (s + 1e-8)

    X = np.stack([sig[:-1], sig[1:]], axis=1)
    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]

    if H1.size == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births = births[mask]
    deaths = deaths[mask]
    lifetimes = lifetimes[mask]

    if len(lifetimes) == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)
    lifetime_sum = float(np.sum(lifetimes))
    lifetime_ratio = lifetimes / (lifetime_sum + 1e-8)
    ph_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin = float(np.min(births))
    bmax = float(np.max(births))
    if bmax - bmin < 1e-8:
        betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        betti_entropy = float(entropy(hist + 1e-10))

    avg_persistence_distance = (
        float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0
    )

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (
            2 * np.sum(np.arange(1, n + 1) * lifetimes_sorted)
        ) / (n * np.sum(lifetimes_sorted)) - (n + 1) / n
        gini_index = float(gini)
    else:
        gini_index = 0.0

    lifetime_variance = float(np.var(lifetimes))
    persistent_image_energy = float(np.sum(lifetimes ** 2))

    feats = [
        float(np.mean(births)),                                  # 0
        float(np.std(births)),                                   # 1
        float(np.mean(deaths)),                                  # 2
        float(np.std(deaths)),                                   # 3
        float(np.mean(lifetimes)),                               # 4
        float(np.std(lifetimes)),                                # 5
        float(np.max(lifetimes)),                                # 6
        float(np.min(lifetimes)),                                # 7
        float(np.median(lifetimes)),                             # 8
        float(iqr(lifetimes)),                                   # 9
        safe_skew(lifetimes),                                    # 10
        safe_kurtosis(lifetimes),                                # 11
        float(np.max(births)),                                   # 12
        float(np.min(births)),                                   # 13
        float(np.median(births)),                                # 14
        safe_skew(births),                                       # 15
        safe_kurtosis(births),                                   # 16
        float(np.max(deaths)),                                   # 17
        float(np.min(deaths)),                                   # 18
        float(np.median(deaths)),                                # 19
        safe_skew(deaths),                                       # 20
        safe_kurtosis(deaths),                                   # 21
        float(len(lifetimes)),                                   # 22
        float(lifetimes_sorted[-1]),                             # 23
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,  # 24
        float(np.max(lifetimes) / (np.min(lifetimes) + 1e-8)),   # 25
        ph_entropy,                                              # 26
        betti_entropy,                                           # 27
        avg_persistence_distance,                                # 28
        gini_index,                                              # 29
        lifetime_variance,                                       # 30
        persistent_image_energy,                                 # 31
    ]
    return np.asarray(feats, dtype=np.float32)


# =============================================================================
# Find target blocks
# =============================================================================
def find_target_blocks(labels: np.ndarray):
    labels = np.asarray(labels).reshape(-1)
    if len(labels) == 0:
        return {}

    segments = []
    start = 0
    cur = labels[0]
    for i in range(1, len(labels)):
        if labels[i] != cur:
            segments.append((int(cur), start, i))
            start = i
            cur = labels[i]
    segments.append((int(cur), start, len(labels)))

    baseline_seg = None
    stress_seg = None
    meditation_seg = None

    for lab, s, e in segments:
        if lab == LABEL_BASELINE:
            baseline_seg = (s, e)
            break

    for lab, s, e in segments:
        if lab == LABEL_STRESS:
            stress_seg = (s, e)
            break

    for lab, s, e in reversed(segments):
        if lab == LABEL_MEDITATION:
            meditation_seg = (s, e)
            break

    out = {}
    if baseline_seg is not None:
        out[LABEL_BASELINE] = baseline_seg
    if stress_seg is not None:
        out[LABEL_STRESS] = stress_seg
    if meditation_seg is not None:
        out[LABEL_MEDITATION] = meditation_seg
    return out


def window_fully_in_block(start_l: int, end_l: int, label_major: int, target_blocks: dict) -> bool:
    if label_major not in target_blocks:
        return False
    blk_s, blk_e = target_blocks[label_major]
    return (start_l >= blk_s) and (end_l <= blk_e)


# =============================================================================
# QC metric helpers
# =============================================================================
def is_flatline(sig: np.ndarray, fs: int, eps: float = FLATLINE_EPS, min_sec: float = FLATLINE_MIN_SEC) -> bool:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    run_len = int(round(fs * min_sec))
    if len(sig) < run_len:
        return False
    ds = np.abs(np.diff(sig))
    low_change = ds < eps
    cnt = 0
    for v in low_change:
        if v:
            cnt += 1
            if cnt >= run_len - 1:
                return True
        else:
            cnt = 0
    return False


def clipping_ratio(sig: np.ndarray) -> float:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    if len(sig) == 0:
        return 1.0
    lo = np.percentile(sig, 1)
    hi = np.percentile(sig, 99)
    if hi - lo < 1e-8:
        return 1.0
    near_lo = sig <= (lo + 0.01 * (hi - lo))
    near_hi = sig >= (hi - 0.01 * (hi - lo))
    return float(np.mean(near_lo | near_hi))


def detect_eda_peaks(sig: np.ndarray, fs: int):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    distance = max(1, int(round(MIN_EVENT_DISTANCE_SEC * fs)))
    prom = max(0.03, 0.10 * float(np.std(sig)))
    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def compute_motion_score(acc_xyz: np.ndarray) -> float:
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32)
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3 or len(acc_xyz) < 2:
        return 1.0
    mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    dmag = np.abs(np.diff(mag))
    score = np.median(dmag) + 0.5 * np.std(dmag)
    return float(score)


def compute_acc_diff_metrics(acc_xyz: np.ndarray):
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32)
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3 or len(acc_xyz) < 2:
        return {
            "acc_diff_mean": np.nan,
            "acc_diff_exceed_ratio": np.nan,
            "acc_diff_flag": 1
        }

    mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    dmag = np.abs(np.diff(mag))

    acc_diff_mean = float(np.mean(dmag)) if len(dmag) > 0 else 0.0
    acc_diff_exceed_ratio = float(np.mean(dmag > ACC_DIFF_THRESH)) if len(dmag) > 0 else 0.0
    acc_diff_flag = int(acc_diff_mean > ACC_DIFF_THRESH)

    return {
        "acc_diff_mean": acc_diff_mean,
        "acc_diff_exceed_ratio": acc_diff_exceed_ratio,
        "acc_diff_flag": acc_diff_flag
    }


def _safe_psd_1d(x: np.ndarray, fs: int, fmin: float, fmax: float):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) < 8:
        return None, None

    x = x[np.isfinite(x)]
    if len(x) < 8:
        return None, None

    if np.std(x) < 1e-8:
        return None, None

    nperseg = min(256, len(x))
    f, pxx = welch(x, fs=fs, nperseg=nperseg)

    mask = (f >= fmin) & (f <= min(fmax, fs / 2 - 1e-6))
    if np.sum(mask) < 4:
        return None, None

    return f[mask], pxx[mask]


def _corr_on_common_freq(f_ref, p_ref, f_other, p_other):
    if f_ref is None or f_other is None:
        return np.nan

    p_other_interp = np.interp(f_ref, f_other, p_other)

    a = np.log1p(p_ref)
    b = np.log1p(p_other_interp)

    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan

    r = np.corrcoef(a, b)[0, 1]
    return float(r) if np.isfinite(r) else np.nan


def compute_eda_acc_psd_corr(seg_eda_bp: np.ndarray, seg_acc_xyz: np.ndarray,
                             fs_eda: int, fs_acc: int,
                             fmin: float = PSD_FMIN, fmax: float = PSD_FMAX):
    out = {
        "psd_corr_x": np.nan,
        "psd_corr_y": np.nan,
        "psd_corr_z": np.nan,
        "psd_corr_mag": np.nan,
        "psd_corr_max": np.nan,
        "psd_corr_flag": 1
    }

    seg_eda_bp = np.asarray(seg_eda_bp, dtype=np.float32).reshape(-1)
    acc_xyz = np.asarray(seg_acc_xyz, dtype=np.float32)

    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3:
        return out

    f_eda, p_eda = _safe_psd_1d(seg_eda_bp, fs_eda, fmin, fmax)
    if f_eda is None:
        return out

    acc_mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))

    corr_list = []
    names_and_signals = [
        ("x", acc_xyz[:, 0]),
        ("y", acc_xyz[:, 1]),
        ("z", acc_xyz[:, 2]),
        ("mag", acc_mag),
    ]

    for name, sig in names_and_signals:
        f_acc, p_acc = _safe_psd_1d(sig, fs_acc, fmin, fmax)
        r = _corr_on_common_freq(f_eda, p_eda, f_acc, p_acc)
        out[f"psd_corr_{name}"] = r
        if np.isfinite(r):
            corr_list.append(r)

    if len(corr_list) == 0:
        return out

    psd_corr_max = float(np.max(corr_list))
    out["psd_corr_max"] = psd_corr_max
    out["psd_corr_flag"] = int(psd_corr_max >= PSD_CORR_THRESH)

    return out


def extract_events(sig: np.ndarray, peaks: np.ndarray, fs: int):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    events = []
    before = int(round(BEAT_BEFORE_SEC * fs))
    after = int(round(BEAT_AFTER_SEC * fs))
    target_x = np.linspace(0, 1, BEAT_RESAMPLED_LEN)

    for p in peaks:
        s = p - before
        e = p + after
        if s < 0 or e > len(sig):
            continue
        evt = sig[s:e].copy()
        if len(evt) < 5:
            continue
        src_x = np.linspace(0, 1, len(evt))
        f = interp1d(src_x, evt, kind="linear")
        evt_rs = f(target_x).astype(np.float32)
        events.append(evt_rs)

    if len(events) == 0:
        return np.empty((0, BEAT_RESAMPLED_LEN), dtype=np.float32)
    return np.stack(events, axis=0)


def median_template_corr(beats: np.ndarray) -> float:
    if beats.ndim != 2 or len(beats) < 2:
        return 0.0
    template = np.median(beats, axis=0)
    tc_list = []
    for b in beats:
        sb = np.std(b)
        st = np.std(template)
        if sb < 1e-8 or st < 1e-8:
            tc_list.append(0.0)
        else:
            r = np.corrcoef(b, template)[0, 1]
            if np.isfinite(r):
                tc_list.append(float(r))
            else:
                tc_list.append(0.0)
    return float(np.median(tc_list)) if len(tc_list) > 0 else 0.0


def compute_interval_metrics(peaks: np.ndarray, fs: int):
    peaks = np.asarray(peaks, dtype=np.int64)
    if len(peaks) < 3:
        return {
            "n_peaks": int(len(peaks)),
            "ibi_pass_ratio": 0.0,
            "valid_beat_ratio": 0.0,
            "ibi_plausibility": 0.0,
            "ibi_mean_sec": np.nan,
        }

    ibi = np.diff(peaks) / float(fs)
    plausible = (ibi >= IBI_MIN_SEC) & (ibi <= IBI_MAX_SEC)
    ibi_pass_ratio = float(np.mean(plausible)) if len(ibi) > 0 else 0.0

    if len(ibi) >= 3:
        med_ibi = np.median(ibi)
        valid = plausible & (np.abs(ibi - med_ibi) <= 0.50 * (med_ibi + 1e-8))
        vbr = float(np.mean(valid))
    else:
        vbr = ibi_pass_ratio

    return {
        "n_peaks": int(len(peaks)),
        "ibi_pass_ratio": ibi_pass_ratio,
        "valid_beat_ratio": vbr,
        "ibi_plausibility": ibi_pass_ratio,
        "ibi_mean_sec": float(np.mean(ibi)) if len(ibi) > 0 else np.nan,
    }


def notch_foot_peak_detectability_score(beats: np.ndarray) -> float:
    if beats.ndim != 2 or len(beats) == 0:
        return 0.0

    scores = []
    for b in beats:
        foot = np.min(b[:max(5, len(b)//4)])
        peak = np.max(b)
        amp = peak - foot
        dyn = np.max(b) - np.min(b) + 1e-8
        s = amp / dyn
        scores.append(clamp01(s))

    return float(np.median(scores)) if len(scores) > 0 else 0.0


def compute_sqi_scores(vbr, tc, ibi_plausibility, motion_score, morph_score):
    motion_penalty = clamp01(motion_score)

    sqi_stress_basic = (
        0.40 * clamp01(vbr)
        + 0.30 * clamp01(tc)
        + 0.20 * clamp01(ibi_plausibility)
        - 0.10 * motion_penalty
    )

    sqi_stress_morph = (
        0.30 * clamp01(vbr)
        + 0.25 * clamp01(tc)
        + 0.20 * clamp01(ibi_plausibility)
        + 0.25 * clamp01(morph_score)
        - motion_penalty
    )

    return float(sqi_stress_basic), float(sqi_stress_morph), motion_penalty


def evaluate_window_quality(seg_eda_raw: np.ndarray, seg_acc_xyz: np.ndarray, fs_eda: int, fs_acc: int):
    out = {}

    # missing
    out["bvp_missing_ratio"] = compute_missing_ratio(seg_eda_raw)   # 컬럼명은 원래 구조 유지
    out["acc_missing_ratio"] = compute_missing_ratio(seg_acc_xyz)
    out["missing_flag"] = int(
        (out["bvp_missing_ratio"] > 0.0) or
        (out["acc_missing_ratio"] > 0.0)
    ) if not ALLOW_MISSING else 0

    seg_eda = fill_nan_with_median(seg_eda_raw)
    seg_acc_xyz = fill_nan_acc_with_median(seg_acc_xyz)

    # EDA preprocess
    seg_eda_bp = bandpass_filter(seg_eda, fs=fs_eda, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)
    seg_eda_z = zscore_1d(seg_eda_bp)

    # existing-style QC
    out["flatline_flag"] = int(is_flatline(seg_eda_bp, fs=fs_eda))
    out["clipping_ratio"] = clipping_ratio(seg_eda_bp)
    out["clipping_flag"] = int(out["clipping_ratio"] > CLIPPING_RATIO_MAX)

    peaks, props = detect_eda_peaks(seg_eda_z, fs=fs_eda)
    out["n_peaks"] = int(len(peaks))
    out["peak_sparse_flag"] = int(len(peaks) < MIN_PEAKS_PER_MIN)

    ibi_info = compute_interval_metrics(peaks, fs_eda)
    out.update(ibi_info)

    beats = extract_events(seg_eda_z, peaks, fs_eda)
    tc = median_template_corr(beats)
    morph_score = notch_foot_peak_detectability_score(beats)
    motion_score_raw = compute_motion_score(seg_acc_xyz)

    out["median_template_corr"] = float(tc)
    out["notch_foot_peak_detectability"] = float(morph_score)
    out["motion_score_raw"] = float(motion_score_raw)

    sqi_basic, sqi_morph, motion_penalty = compute_sqi_scores(
        vbr=out["valid_beat_ratio"],
        tc=tc,
        ibi_plausibility=out["ibi_plausibility"],
        motion_score=motion_score_raw,
        morph_score=morph_score
    )

    out["motion_score"] = float(motion_penalty)
    out["SQI_stress_basic"] = float(sqi_basic)
    out["SQI_stress_morph"] = float(sqi_morph)

    # motion artifact QC
    acc_diff_info = compute_acc_diff_metrics(seg_acc_xyz)
    out.update(acc_diff_info)

    psd_corr_info = compute_eda_acc_psd_corr(
        seg_eda_bp=seg_eda_bp,
        seg_acc_xyz=seg_acc_xyz,
        fs_eda=fs_eda,
        fs_acc=fs_acc,
        fmin=PSD_FMIN,
        fmax=PSD_FMAX
    )
    out.update(psd_corr_info)

    hard_reject = (
        (out["missing_flag"] == 1) or
        (out["flatline_flag"] == 1) or
        (out["clipping_flag"] == 1) or
        (out["peak_sparse_flag"] == 1) or
        (out["acc_diff_flag"] == 1) or
        (out["psd_corr_flag"] == 1)
    )
    out["hard_reject"] = int(hard_reject)

    return seg_eda_z, out


# =============================================================================
# Build table for one subject
# =============================================================================
def build_subject_windows_table(pkl_path: str, subject_name: str) -> pd.DataFrame:
    with open(pkl_path, "rb") as f:
        s = pickle.load(f, encoding="latin1")

    eda = np.asarray(s["signal"]["wrist"]["EDA"]).reshape(-1).astype(np.float32)
    acc = np.asarray(s["signal"]["wrist"]["ACC"]).astype(np.float32)
    labels = np.asarray(s["label"]).reshape(-1).astype(np.int64)

    dur_wrist = len(eda) / FS_EDA
    labels = labels[: int(dur_wrist * FS_LABEL)]

    eda = fill_nan_with_median(eda)

    target_blocks = find_target_blocks(labels)
    if len(target_blocks) == 0:
        return pd.DataFrame()

    Wl = int(WINDOW_SEC * FS_LABEL)
    Sl = int(STEP_SEC * FS_LABEL)
    We = int(WINDOW_SEC * FS_EDA)
    Wa = int(WINDOW_SEC * FS_ACC)

    rows = []
    window_counter = 0

    for start_l in range(0, len(labels) - Wl + 1, Sl):
        end_l = start_l + Wl
        win_lab = labels[start_l:end_l]

        binc = np.bincount(win_lab)
        maj = int(np.argmax(binc))
        maj_ratio = float((win_lab == maj).mean())

        if maj not in [LABEL_BASELINE, LABEL_STRESS, LABEL_MEDITATION]:
            continue
        if maj_ratio < 0.95:
            continue
        if not window_fully_in_block(start_l, end_l, maj, target_blocks):
            continue

        t0 = start_l / FS_LABEL
        t1 = t0 + WINDOW_SEC

        start_e = int(round(t0 * FS_EDA))
        end_e = start_e + We
        start_a = int(round(t0 * FS_ACC))
        end_a = start_a + Wa

        if end_e > len(eda) or end_a > len(acc):
            continue

        seg_eda_raw = eda[start_e:end_e]
        seg_acc_xyz = acc[start_a:end_a]

        if len(seg_eda_raw) != We or len(seg_acc_xyz) != Wa:
            continue

        seg_eda_for_tpv, qc_info = evaluate_window_quality(seg_eda_raw, seg_acc_xyz, FS_EDA, FS_ACC)
        tpv_full = extract_tpv_features(seg_eda_for_tpv)

        window_counter += 1

        row = {
            "subject": subject_name,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "status": STATUS_MAP[maj],
            "status_name": STATUS_NAME_MAP[maj],
            "label_major": maj,
            "t_start_sec": float(t0),
            "t_end_sec": float(t1),
            "major_ratio": maj_ratio,
        }

        for k, v in qc_info.items():
            row[k] = v

        for idx in SELECTED_TPV_INDICES:
            row[f"TPV{idx}"] = float(tpv_full[idx])

        rows.append(row)

    return pd.DataFrame(rows)


# =============================================================================
# Build all subjects
# =============================================================================
def build_all_subjects_windows_table(pkl_dir: str) -> pd.DataFrame:
    pkl_list = sorted(glob.glob(os.path.join(pkl_dir, "S*.pkl")))
    all_dfs = []

    print(f"[INFO] Found {len(pkl_list)} pkl files")

    for pkl_path in pkl_list:
        subject_name = os.path.splitext(os.path.basename(pkl_path))[0]
        print(f"\n[INFO] Processing {subject_name} ...")
        try:
            df_sub = build_subject_windows_table(pkl_path, subject_name)
            print(f"[INFO] {subject_name}: extracted {len(df_sub)} target windows")
            if len(df_sub) > 0:
                all_dfs.append(df_sub)
        except Exception as e:
            print(f"[WARN] Failed on {subject_name}: {e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    return pd.concat(all_dfs, axis=0, ignore_index=True)


# =============================================================================
# Personalized QC pass/fail
# =============================================================================
def apply_relaxed_personalized_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    motion_thr_map = (
        df.groupby("subject")["motion_score"]
        .quantile(MOTION_PERCENTILE / 100.0)
        .to_dict()
    )

    df["motion_threshold_subject_p80"] = df["subject"].map(motion_thr_map)

    df["qc_relaxed_pass"] = (
        (df["hard_reject"] == 0) &
        (df["valid_beat_ratio"] >= VBR_MIN) &
        (df["median_template_corr"] >= TC_MIN) &
        (df["ibi_pass_ratio"] >= IBI_PASS_MIN) &
        (df["motion_score"] <= df["motion_threshold_subject_p80"])
    ).astype(int)

    return df


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    df_all = build_all_subjects_windows_table(PKL_DIR)

    print("\n[INFO] final raw shape:", df_all.shape)
    if len(df_all) == 0:
        print("[WARN] No target windows extracted.")
        raise SystemExit

    df_all = apply_relaxed_personalized_qc(df_all)

    base_cols = [
        "subject", "window", "window_index",
        "status", "status_name", "label_major",
        "t_start_sec", "t_end_sec", "major_ratio"
    ]

    qc_cols = [
        "bvp_missing_ratio", "acc_missing_ratio", "missing_flag",
        "flatline_flag", "clipping_ratio", "clipping_flag",
        "n_peaks", "peak_sparse_flag",
        "ibi_pass_ratio", "valid_beat_ratio", "ibi_plausibility", "ibi_mean_sec",
        "median_template_corr", "notch_foot_peak_detectability",

        "acc_diff_mean", "acc_diff_exceed_ratio", "acc_diff_flag",

        "psd_corr_x", "psd_corr_y", "psd_corr_z", "psd_corr_mag",
        "psd_corr_max", "psd_corr_flag",

        "motion_score_raw", "motion_score",
        "SQI_stress_basic", "SQI_stress_morph",
        "hard_reject", "motion_threshold_subject_p80", "qc_relaxed_pass"
    ]

    feat_cols = [f"TPV{idx}" for idx in SELECTED_TPV_INDICES]

    final_cols = base_cols + qc_cols + feat_cols
    df_all = df_all[final_cols].copy()

    # 1) save all target windows (no QC filtering)
    csv_all = os.path.join(OUTPUT_DIR, "noQC_EDA.csv")
    df_all.to_csv(csv_all, index=False)
    print(f"[INFO] Saved no-QC CSV: {csv_all}")

    # 2) save only QC-passed windows
    df_qc = df_all[df_all["qc_relaxed_pass"] == 1].copy()
    csv_qc = os.path.join(OUTPUT_DIR, "withQC_EDA.csv")
    df_qc.to_csv(csv_qc, index=False)
    print(f"[INFO] Saved QC CSV: {csv_qc}")

    # 3) subject/status summary
    summary = (
        df_all.groupby(["subject", "status_name"])
        .agg(
            n_total=("window", "count"),
            n_qc_pass=("qc_relaxed_pass", "sum")
        )
        .reset_index()
    )
    summary["retention_ratio"] = summary["n_qc_pass"] / summary["n_total"]

    summary_csv = os.path.join(OUTPUT_DIR, "summary_EDA.csv")
    summary.to_csv(summary_csv, index=False)
    print(f"[INFO] Saved QC retention summary: {summary_csv}")

    print("\n[INFO] Status counts (all target windows)")
    print(df_all["status_name"].value_counts())

    print("\n[INFO] Status counts (QC passed)")
    print(df_qc["status_name"].value_counts())

    print("\n[INFO] Subject x status retention summary")
    print(summary.to_string(index=False))

[INFO] Found 15 pkl files

[INFO] Processing S10 ...
[INFO] S10: extracted 36 target windows

[INFO] Processing S11 ...
[INFO] S11: extracted 35 target windows

[INFO] Processing S13 ...
[INFO] S13: extracted 35 target windows

[INFO] Processing S14 ...
[INFO] S14: extracted 35 target windows

[INFO] Processing S15 ...
[INFO] S15: extracted 35 target windows

[INFO] Processing S16 ...
[INFO] S16: extracted 35 target windows

[INFO] Processing S17 ...
[INFO] S17: extracted 36 target windows

[INFO] Processing S2 ...
[INFO] S2: extracted 33 target windows

[INFO] Processing S3 ...
[INFO] S3: extracted 34 target windows

[INFO] Processing S4 ...
[INFO] S4: extracted 34 target windows

[INFO] Processing S5 ...
[INFO] S5: extracted 33 target windows

[INFO] Processing S6 ...
[INFO] S6: extracted 34 target windows

[INFO] Processing S7 ...
[INFO] S7: extracted 35 target windows

[INFO] Processing S8 ...
[INFO] S8: extracted 36 target windows

[INFO] Processing S9 ...
[INFO] S9: extracted 32 